In [43]:
import pandas as pd
import numpy as np

In [44]:
df = pd.read_csv('akiya_data.csv')

In [45]:
df = df.drop(columns = 'Image')

In [46]:
df = df.drop_duplicates()

In [47]:
df[['Store', 'Minute_Walk']] = df['Konbini'].str.split('(', expand=True)

In [48]:
df

,Location,Price,Type,Beds,Land,House,Konbini,Store,Minute_Walk
0,"Ichigawa, Chiba","$170,000","Apartment, Land",3,-,65m²,Family Mart (4 min),Family Mart,4 min)
1,"Funabashi, Chiba","$180,000",House,3,96m²,93m²,Seven Eleven (6 min),Seven Eleven,6 min)
2,"Funabashi, Chiba","$180,000",House,3,128m²,93m²,Family Mart (2 min),Family Mart,2 min)
3,"Funabashi, Chiba","$190,000","House, Land",5,136m²,132m²,Lawson (6 min),Lawson,6 min)
4,"Funabashi, Chiba","$190,000",House,2,164m²,91m²,Lawson (4 min),Lawson,4 min)
...,...,...,...,...,...,...,...,...,...
6355,"Kita, Hokkaido","$94,000",House,5,188m²,140m²,Seven Eleven (4 min),Seven Eleven,4 min)
6356,"Kita, Hokkaido","$99,000",House,2,209m²,58m²,Seicomart (7 min),Seicomart,7 min)
6357,"Kita, Hokkaido","$99,000",House,4,183m²,94m²,Seven Eleven (6 min),Seven Eleven,6 min)
6358,"Minato, Tokyo","$620,000","Apartment, Land",1,-,55m²,Lawson (1 min),Lawson,1 min)


In [49]:
# 1. Extract the digits safely
extracted_digits = df['Minute_Walk'].str.extract(r'(\d+)')

# 2. Fill missing/empty rows with 0 so the conversion doesn't crash, then cast to int
df['Minute_Walk'] = extracted_digits.fillna(0).astype(int)

In [50]:
df['House'] = df['House'].str.replace('m²', '').str.strip()

In [51]:
df = df.rename(columns={'House': 'House (m2)'})

In [52]:
df = df.rename(columns={'Land': 'Land (m2)'})

In [53]:
df['Land (m2)'] = df['Land (m2)'].str.replace('m²', '').str.strip()

In [54]:
df

,Location,Price,Type,Beds,Land (m2),House (m2),Konbini,Store,Minute_Walk
0,"Ichigawa, Chiba","$170,000","Apartment, Land",3,-,65,Family Mart (4 min),Family Mart,4
1,"Funabashi, Chiba","$180,000",House,3,96,93,Seven Eleven (6 min),Seven Eleven,6
2,"Funabashi, Chiba","$180,000",House,3,128,93,Family Mart (2 min),Family Mart,2
3,"Funabashi, Chiba","$190,000","House, Land",5,136,132,Lawson (6 min),Lawson,6
4,"Funabashi, Chiba","$190,000",House,2,164,91,Lawson (4 min),Lawson,4
...,...,...,...,...,...,...,...,...,...
6355,"Kita, Hokkaido","$94,000",House,5,188,140,Seven Eleven (4 min),Seven Eleven,4
6356,"Kita, Hokkaido","$99,000",House,2,209,58,Seicomart (7 min),Seicomart,7
6357,"Kita, Hokkaido","$99,000",House,4,183,94,Seven Eleven (6 min),Seven Eleven,6
6358,"Minato, Tokyo","$620,000","Apartment, Land",1,-,55,Lawson (1 min),Lawson,1


In [55]:
df.to_csv('akiya_data_cleaned.csv', index=False)

# 2nd Phase

In [56]:
fd = pd.read_csv('akiya_data_cleaned.csv')

In [57]:
# Wrap the condition in parentheses so Python evaluates it first
(fd == '-').sum()

Location          0
Price             0
Type              0
Beds            384
Land (m2)      1785
House (m2)       97
Konbini         146
Store           146
Minute_Walk       0
dtype: int64

In [58]:
fd = fd.replace('-', np.nan)

In [59]:
fd.isna().sum()

Location          2
Price             0
Type              0
Beds            384
Land (m2)      1785
House (m2)       97
Konbini         146
Store           146
Minute_Walk       0
dtype: int64

In [60]:
#turns the columns into numeric, coercing errors to NaN (which will be ignored in calculations)
#turning string into nummeric
fd['Beds'] = pd.to_numeric(fd['Beds'], errors='coerce')
fd['Land (m2)'] = pd.to_numeric(fd['Land (m2)'], errors='coerce')
fd['House (m2)'] = pd.to_numeric(fd['House (m2)'], errors='coerce')
fd['Minute_Walk'] = pd.to_numeric(fd['Minute_Walk'], errors='coerce')

In [61]:
fd['Beds'] = fd['Beds'].fillna(fd['Beds'].median())
fd['Land (m2)'] = fd['Land (m2)'].fillna(fd['Land (m2)'].median())
fd['House (m2)'] = fd['House (m2)'].fillna(fd['House (m2)'].median())
fd['Minute_Walk'] = fd['Minute_Walk'].fillna(fd['Minute_Walk'].median())
fd = fd.drop(columns=['Konbini', 'Store'])  # drop the text columns, keep only Minute_Walk
fd = fd.dropna(subset=['Location'])

In [62]:
fd

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk
0,"Ichigawa, Chiba","$170,000","Apartment, Land",3.0,192.0,65.0,4
1,"Funabashi, Chiba","$180,000",House,3.0,96.0,93.0,6
2,"Funabashi, Chiba","$180,000",House,3.0,128.0,93.0,2
3,"Funabashi, Chiba","$190,000","House, Land",5.0,136.0,132.0,6
4,"Funabashi, Chiba","$190,000",House,2.0,164.0,91.0,4
...,...,...,...,...,...,...,...
6175,"Kita, Hokkaido","$94,000",House,5.0,188.0,140.0,4
6176,"Kita, Hokkaido","$99,000",House,2.0,209.0,58.0,7
6177,"Kita, Hokkaido","$99,000",House,4.0,183.0,94.0,6
6178,"Minato, Tokyo","$620,000","Apartment, Land",1.0,192.0,55.0,1


In [ ]:
fd['Price'] = fd['Price'].str.replace(',', '').str.strip()
fd['Price'] = fd['Price'].str.replace('$', '', regex=False)
fd['Price'] = pd.to_numeric(fd['Price'], errors='coerce')
# 2. Convert the column to an integer so the model can read it as a target vector

In [69]:
fd

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk
0,"Ichigawa, Chiba",170000.0,"Apartment, Land",3.0,192.0,65.0,4
1,"Funabashi, Chiba",180000.0,House,3.0,96.0,93.0,6
2,"Funabashi, Chiba",180000.0,House,3.0,128.0,93.0,2
3,"Funabashi, Chiba",190000.0,"House, Land",5.0,136.0,132.0,6
4,"Funabashi, Chiba",190000.0,House,2.0,164.0,91.0,4
...,...,...,...,...,...,...,...
6175,"Kita, Hokkaido",94000.0,House,5.0,188.0,140.0,4
6176,"Kita, Hokkaido",99000.0,House,2.0,209.0,58.0,7
6177,"Kita, Hokkaido",99000.0,House,4.0,183.0,94.0,6
6178,"Minato, Tokyo",620000.0,"Apartment, Land",1.0,192.0,55.0,1


In [70]:
fd = fd.dropna(subset=['Price'])

In [71]:
# Keep only rows where the 'Type' column contains the word 'House'
# na=False prevents the code from crashing if there are blank/NaN values in the column
fd = fd[fd['Type'].str.contains('House', na=False)]

In [72]:
fd

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk
1,"Funabashi, Chiba",180000.0,House,3.0,96.0,93.0,6
2,"Funabashi, Chiba",180000.0,House,3.0,128.0,93.0,2
3,"Funabashi, Chiba",190000.0,"House, Land",5.0,136.0,132.0,6
4,"Funabashi, Chiba",190000.0,House,2.0,164.0,91.0,4
5,"Funabashi, Chiba",190000.0,House,4.0,100.0,98.0,0
...,...,...,...,...,...,...,...
6173,"Kita, Hokkaido",92000.0,House,4.0,183.0,108.0,4
6175,"Kita, Hokkaido",94000.0,House,5.0,188.0,140.0,4
6176,"Kita, Hokkaido",99000.0,House,2.0,209.0,58.0,7
6177,"Kita, Hokkaido",99000.0,House,4.0,183.0,94.0,6


In [73]:
# Extract prefecture from Location (e.g. "Shinagawa, Tokyo" → "Tokyo")
fd['Prefecture'] = fd['Location'].str.split(',').str[-1].str.strip()

# Check value counts before encoding
print(fd['Prefecture'].value_counts())

Prefecture
Hokkaido     1137
Oita          469
Saitama       433
Kagawa        352
Ehime         311
Kanagawa      306
Chiba         226
Kochi         176
Tokushima     167
Tokyo          94
Okinawa        66
Aichi          51
Osaka          44
Ibaraki        36
Tochigi        28
Hyogo          26
Miyagi         25
Akita          24
Gunma          24
Fukuoka        24
Fukushima      22
Niigata        17
Kyoto          17
Yamagata       17
Shizuoka       16
Hiroshima      15
Nagano         14
Wakayama       11
Kumamoto        8
Aomori          8
Mie             5
Iwate           5
Kagoshima       5
Nara            4
Yamaguchi       4
Tottori         4
Yamanashi       3
Saga            3
Fukui           2
Toyama          1
Okayama         1
Ishikawa        1
Shiga           1
Gifu            1
Shimane         1
Name: count, dtype: int64


In [74]:
fd

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk,Prefecture
1,"Funabashi, Chiba",180000.0,House,3.0,96.0,93.0,6,Chiba
2,"Funabashi, Chiba",180000.0,House,3.0,128.0,93.0,2,Chiba
3,"Funabashi, Chiba",190000.0,"House, Land",5.0,136.0,132.0,6,Chiba
4,"Funabashi, Chiba",190000.0,House,2.0,164.0,91.0,4,Chiba
5,"Funabashi, Chiba",190000.0,House,4.0,100.0,98.0,0,Chiba
...,...,...,...,...,...,...,...,...
6173,"Kita, Hokkaido",92000.0,House,4.0,183.0,108.0,4,Hokkaido
6175,"Kita, Hokkaido",94000.0,House,5.0,188.0,140.0,4,Hokkaido
6176,"Kita, Hokkaido",99000.0,House,2.0,209.0,58.0,7,Hokkaido
6177,"Kita, Hokkaido",99000.0,House,4.0,183.0,94.0,6,Hokkaido


In [75]:
fd.to_csv('akiya_data_final.csv', index=False)

In [76]:
df = pd.read_csv('akiya_data_final.csv')

In [77]:
df

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk,Prefecture
0,"Funabashi, Chiba",180000.0,House,3.0,96.0,93.0,6,Chiba
1,"Funabashi, Chiba",180000.0,House,3.0,128.0,93.0,2,Chiba
2,"Funabashi, Chiba",190000.0,"House, Land",5.0,136.0,132.0,6,Chiba
3,"Funabashi, Chiba",190000.0,House,2.0,164.0,91.0,4,Chiba
4,"Funabashi, Chiba",190000.0,House,4.0,100.0,98.0,0,Chiba
...,...,...,...,...,...,...,...,...
4200,"Kita, Hokkaido",92000.0,House,4.0,183.0,108.0,4,Hokkaido
4201,"Kita, Hokkaido",94000.0,House,5.0,188.0,140.0,4,Hokkaido
4202,"Kita, Hokkaido",99000.0,House,2.0,209.0,58.0,7,Hokkaido
4203,"Kita, Hokkaido",99000.0,House,4.0,183.0,94.0,6,Hokkaido


In [78]:
top_prefectures = fd['Prefecture'].value_counts()
top_prefectures = top_prefectures[top_prefectures >= 10].index

fd['Prefecture'] = fd['Prefecture'].apply(
    lambda x: x if x in top_prefectures else 'Other'
)

In [79]:
fd

,Location,Price,Type,Beds,Land (m2),House (m2),Minute_Walk,Prefecture
1,"Funabashi, Chiba",180000.0,House,3.0,96.0,93.0,6,Chiba
2,"Funabashi, Chiba",180000.0,House,3.0,128.0,93.0,2,Chiba
3,"Funabashi, Chiba",190000.0,"House, Land",5.0,136.0,132.0,6,Chiba
4,"Funabashi, Chiba",190000.0,House,2.0,164.0,91.0,4,Chiba
5,"Funabashi, Chiba",190000.0,House,4.0,100.0,98.0,0,Chiba
...,...,...,...,...,...,...,...,...
6173,"Kita, Hokkaido",92000.0,House,4.0,183.0,108.0,4,Hokkaido
6175,"Kita, Hokkaido",94000.0,House,5.0,188.0,140.0,4,Hokkaido
6176,"Kita, Hokkaido",99000.0,House,2.0,209.0,58.0,7,Hokkaido
6177,"Kita, Hokkaido",99000.0,House,4.0,183.0,94.0,6,Hokkaido


In [80]:
fd['has_house'] = fd['Type'].str.contains('House').astype(int)
fd['has_apartment'] = fd['Type'].str.contains('Apartment').astype(int)
fd['has_land'] = fd['Type'].str.contains('Land').astype(int)

# Drop the original Type column
fd = fd.drop(columns=['Type'])

In [81]:
fd = fd.drop(columns=['Location'])

In [82]:
fd

,Price,Beds,Land (m2),House (m2),Minute_Walk,Prefecture,has_house,has_apartment,has_land
1,180000.0,3.0,96.0,93.0,6,Chiba,1,0,0
2,180000.0,3.0,128.0,93.0,2,Chiba,1,0,0
3,190000.0,5.0,136.0,132.0,6,Chiba,1,0,1
4,190000.0,2.0,164.0,91.0,4,Chiba,1,0,0
5,190000.0,4.0,100.0,98.0,0,Chiba,1,0,0
...,...,...,...,...,...,...,...,...,...
6173,92000.0,4.0,183.0,108.0,4,Hokkaido,1,0,0
6175,94000.0,5.0,188.0,140.0,4,Hokkaido,1,0,0
6176,99000.0,2.0,209.0,58.0,7,Hokkaido,1,0,0
6177,99000.0,4.0,183.0,94.0,6,Hokkaido,1,0,0


In [83]:
# Convert all boolean columns to int (True→1, False→0)
bool_cols = fd.select_dtypes(include='bool').columns
fd[bool_cols] = fd[bool_cols].astype(int)

print(fd.head())
print(fd.shape)

      Price  Beds  Land (m2)  House (m2)  Minute_Walk Prefecture  has_house  \
1  180000.0   3.0       96.0        93.0            6      Chiba          1   
2  180000.0   3.0      128.0        93.0            2      Chiba          1   
3  190000.0   5.0      136.0       132.0            6      Chiba          1   
4  190000.0   2.0      164.0        91.0            4      Chiba          1   
5  190000.0   4.0      100.0        98.0            0      Chiba          1   

   has_apartment  has_land  
1              0         0  
2              0         0  
3              0         1  
4              0         0  
5              0         0  
(4205, 9)


In [84]:
fd.info()
print("\nMissing values per column:")
print(fd.isna().sum())

<class 'pandas.DataFrame'>
Index: 4205 entries, 1 to 6179
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Price          4205 non-null   float64
 1   Beds           4205 non-null   float64
 2   Land (m2)      4205 non-null   float64
 3   House (m2)     4205 non-null   float64
 4   Minute_Walk    4205 non-null   int64  
 5   Prefecture     4205 non-null   str    
 6   has_house      4205 non-null   int64  
 7   has_apartment  4205 non-null   int64  
 8   has_land       4205 non-null   int64  
dtypes: float64(4), int64(4), str(1)
memory usage: 355.3 KB

Missing values per column:
Price            0
Beds             0
Land (m2)        0
House (m2)       0
Minute_Walk      0
Prefecture       0
has_house        0
has_apartment    0
has_land         0
dtype: int64


In [85]:
fd

,Price,Beds,Land (m2),House (m2),Minute_Walk,Prefecture,has_house,has_apartment,has_land
1,180000.0,3.0,96.0,93.0,6,Chiba,1,0,0
2,180000.0,3.0,128.0,93.0,2,Chiba,1,0,0
3,190000.0,5.0,136.0,132.0,6,Chiba,1,0,1
4,190000.0,2.0,164.0,91.0,4,Chiba,1,0,0
5,190000.0,4.0,100.0,98.0,0,Chiba,1,0,0
...,...,...,...,...,...,...,...,...,...
6173,92000.0,4.0,183.0,108.0,4,Hokkaido,1,0,0
6175,94000.0,5.0,188.0,140.0,4,Hokkaido,1,0,0
6176,99000.0,2.0,209.0,58.0,7,Hokkaido,1,0,0
6177,99000.0,4.0,183.0,94.0,6,Hokkaido,1,0,0


In [86]:
fd.to_csv('akiya_data_final.csv', index=False)

In [87]:
sf = pd.read_csv('akiya_data_final.csv')

In [89]:
sf

,Price,Beds,Land (m2),House (m2),Minute_Walk,Prefecture,has_house,has_apartment,has_land
0,180000.0,3.0,96.0,93.0,6,Chiba,1,0,0
1,180000.0,3.0,128.0,93.0,2,Chiba,1,0,0
2,190000.0,5.0,136.0,132.0,6,Chiba,1,0,1
3,190000.0,2.0,164.0,91.0,4,Chiba,1,0,0
4,190000.0,4.0,100.0,98.0,0,Chiba,1,0,0
...,...,...,...,...,...,...,...,...,...
4200,92000.0,4.0,183.0,108.0,4,Hokkaido,1,0,0
4201,94000.0,5.0,188.0,140.0,4,Hokkaido,1,0,0
4202,99000.0,2.0,209.0,58.0,7,Hokkaido,1,0,0
4203,99000.0,4.0,183.0,94.0,6,Hokkaido,1,0,0


In [90]:
# Check prefecture distribution first
print(sf['Prefecture'].value_counts())

# Group rare prefectures into "Other" (less than 10 rows)
top_prefectures = sf['Prefecture'].value_counts()
top_prefectures = top_prefectures[top_prefectures >= 10].index

sf['Prefecture'] = sf['Prefecture'].apply(
    lambda x: x if x in top_prefectures else 'Other'
)

# One-hot encode
sf = pd.get_dummies(sf, columns=['Prefecture'], prefix='pref', dtype=int)

print(sf.shape)

Prefecture
Hokkaido     1137
Oita          469
Saitama       433
Kagawa        352
Ehime         311
Kanagawa      306
Chiba         226
Kochi         176
Tokushima     167
Tokyo          94
Okinawa        66
Other          57
Aichi          51
Osaka          44
Ibaraki        36
Tochigi        28
Hyogo          26
Miyagi         25
Akita          24
Gunma          24
Fukuoka        24
Fukushima      22
Niigata        17
Kyoto          17
Yamagata       17
Shizuoka       16
Hiroshima      15
Nagano         14
Wakayama       11
Name: count, dtype: int64
(4205, 37)


In [93]:
# Filter house only
sf = sf[sf['has_house'] == 1]

# Drop type columns
sf = sf.drop(columns=['has_house', 'has_apartment', 'has_land'])

# Drop Location if still present
if 'Location' in sf.columns:
    sf = sf.drop(columns=['Location'])

# Convert any boolean columns to int
bool_cols = sf.select_dtypes(include='bool').columns
sf[bool_cols] = sf[bool_cols].astype(int)

print(sf.shape)
print(fd.isna().sum())

(4205, 34)
Price          0
Beds           0
Land (m2)      0
House (m2)     0
Minute_Walk    0
Prefecture     0
dtype: int64


In [92]:
sf

,Price,Beds,Land (m2),House (m2),Minute_Walk,has_house,has_apartment,has_land,pref_Aichi,pref_Akita,...,pref_Okinawa,pref_Osaka,pref_Other,pref_Saitama,pref_Shizuoka,pref_Tochigi,pref_Tokushima,pref_Tokyo,pref_Wakayama,pref_Yamagata
0,180000.0,3.0,96.0,93.0,6,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,180000.0,3.0,128.0,93.0,2,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,190000.0,5.0,136.0,132.0,6,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,190000.0,2.0,164.0,91.0,4,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,190000.0,4.0,100.0,98.0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4200,92000.0,4.0,183.0,108.0,4,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4201,94000.0,5.0,188.0,140.0,4,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4202,99000.0,2.0,209.0,58.0,7,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4203,99000.0,4.0,183.0,94.0,6,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [94]:
sf.to_csv('cleaned_akiya.csv', index=False)